# Agile Predict — Model Lab
Experiment notebook for testing XGBoost variants locally against production data.
Run `pull_prod_data.ps1` first to generate the CSVs in `data/`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import xgboost as xg
from sklearn.model_selection import cross_val_score

DATA_DIR = Path('data')
PARQUET = DATA_DIR / 'joined.parquet'

# Region B factors (from regions.py)
REGION_MULT = 0.20
REGION_PEAK_OFFSET = 14.0

def agile_to_day_ahead(series: pd.Series) -> pd.Series:
    """Convert Agile retail prices to wholesale day-ahead proxy (region B)."""
    result = series.copy().astype(float)
    idx_gb = pd.to_datetime(result.index).tz_convert('Europe/London')
    peak = (idx_gb.hour >= 16) & (idx_gb.hour < 19)
    result[peak] = result[peak] - REGION_PEAK_OFFSET
    result = result / REGION_MULT
    return result

print('Libraries loaded.')

## Cell 1 — Load & join data
Replicates the exact join logic from `day_ahead_xgb.py`. Uses parquet cache if available.

In [ ]:
if PARQUET.exists():
    df = pd.read_parquet(PARQUET)
    print(f'Loaded from cache: {PARQUET}')
else:
    # ── Load raw tables ──────────────────────────────────────────────────────
    forecasts = pd.read_csv(DATA_DIR / 'forecasts.csv', parse_dates=['created_at'])
    forecasts['created_at'] = pd.to_datetime(forecasts['created_at'], utc=True)
    # Filter seed/bootstrap forecasts (same logic as day_ahead_xgb.py)
    forecasts = forecasts[~forecasts['name'].str.contains('seed|bootstrap', case=False, na=False)].copy()

    fd = pd.read_csv(DATA_DIR / 'forecast_data.csv', parse_dates=['date_time'])
    fd['date_time'] = pd.to_datetime(fd['date_time'], utc=True)

    agile_raw = pd.read_csv(DATA_DIR / 'agile_actual.csv', parse_dates=['date_time'])
    agile_raw['date_time'] = pd.to_datetime(agile_raw['date_time'], utc=True)
    agile_b = agile_raw[agile_raw['region'] == 'B'].copy()
    agile_b = agile_b.set_index('date_time').sort_index()
    agile_b['day_ahead'] = agile_to_day_ahead(agile_b['agile_actual'])

    nordpool = pd.read_csv(DATA_DIR / 'price_history.csv', parse_dates=['date_time'])
    nordpool['date_time'] = pd.to_datetime(nordpool['date_time'], utc=True)
    nordpool = nordpool.set_index('date_time').sort_index()

    # ── Build combined prices: prefer Agile actuals, fill gaps with Nordpool ──
    prices = pd.DataFrame(index=agile_b.index)
    prices['day_ahead'] = agile_b['day_ahead']
    prices['day_ahead'] = prices['day_ahead'].fillna(nordpool['day_ahead'])
    prices = prices.dropna()
    print(f'Price points: {len(prices)} ({prices.index.min().date()} to {prices.index.max().date()})')

    # ── Merge forecasts with forecast_data ───────────────────────────────────
    forecasts_idx = forecasts.set_index('id')[['created_at', 'name']]
    df = fd.merge(forecasts_idx, left_on='forecast_id', right_index=True)
    df = df.set_index('date_time').sort_index()

    # ── Derived time features ────────────────────────────────────────────────
    df['time_gb'] = df.index.tz_convert('Europe/London').hour + df.index.tz_convert('Europe/London').minute / 60
    df['weekend'] = (df.index.day_of_week >= 5).astype(int)
    df['peak'] = ((df['time_gb'] >= 16) & (df['time_gb'] < 19)).astype(float)
    df['sin_hour'] = np.sin(2 * np.pi * df['time_gb'] / 24)
    df['cos_hour'] = np.cos(2 * np.pi * df['time_gb'] / 24)
    df['days_ago'] = (pd.Timestamp.now(tz='UTC') - df['created_at']).dt.total_seconds() / 86400

    # ── Identify train/test split (same as production: 16:15 forecast = train) 
    forecasts['date_gb'] = forecasts['created_at'].dt.tz_convert('Europe/London').dt.normalize()
    forecasts['dt1615'] = (
        (forecasts['date_gb'] + pd.Timedelta(hours=16, minutes=15) - forecasts['created_at'].dt.tz_convert('Europe/London'))
        .dt.total_seconds().abs()
    )
    train_forecast_ids = set(
        forecasts.sort_values('dt1615').drop_duplicates('date_gb')['id']
    )
    df['split'] = df['forecast_id'].apply(lambda x: 'train' if x in train_forecast_ids else 'test')

    # ── Join price target ────────────────────────────────────────────────────
    df = df.join(prices[['day_ahead']], how='inner')
    df = df.dropna(subset=['day_ahead'])

    # ── Filter to training window (same as production: ag_start to ag_end) ──
    fc_windows = forecasts.set_index('id')[['created_at']].copy()
    fc_windows['ag_start'] = fc_windows['created_at'].dt.normalize().dt.tz_convert('UTC') + pd.Timedelta(hours=22)
    fc_windows['ag_end'] = fc_windows['ag_start'] + pd.Timedelta(hours=312)  # ~13 days
    df = df.merge(fc_windows[['ag_start', 'ag_end']], left_on='forecast_id', right_index=True)
    df = df[(df.index >= df['ag_start']) & (df.index < df['ag_end'])]

    df.to_parquet(PARQUET)
    print(f'Saved to {PARQUET}')

print(f'Total rows: {len(df):,}  |  train: {(df.split=="train").sum():,}  |  test: {(df.split=="test").sum():,}')
print(f'Date range: {df.index.min().date()} → {df.index.max().date()}')
print(f'Target (day_ahead): mean={df.day_ahead.mean():.1f}  std={df.day_ahead.std():.1f}  '
      f'p5={df.day_ahead.quantile(0.05):.1f}  p95={df.day_ahead.quantile(0.95):.1f}')

## Cell 2 — Data overview

In [ ]:
# Price time-series coloured by year
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

train_df = df[df['split'] == 'train'].copy()
by_year = train_df.groupby(train_df.index.year)
for year, grp in by_year:
    daily = grp['day_ahead'].resample('D').mean()
    axes[0].plot(daily.index, daily.values, label=str(year), alpha=0.8, linewidth=1)
axes[0].set_title('Day-ahead price (daily mean, training set)')
axes[0].set_ylabel('p/kWh (wholesale)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Mean price by hour-of-day
hourly = train_df.groupby('time_gb')['day_ahead'].mean()
axes[1].bar(hourly.index, hourly.values, width=0.4, color='steelblue', alpha=0.8)
axes[1].axvspan(15.5, 22, alpha=0.1, color='red', label='Evening window (15:30-22:00)')
axes[1].set_title('Mean day-ahead price by hour (training set)')
axes[1].set_xlabel('Hour (GB time)')
axes[1].set_ylabel('p/kWh')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print('Missing values per column:')
print(df.isnull().sum()[df.isnull().sum() > 0])

## Cell 3 — Baseline model (current production config)

In [ ]:
LEGACY_FEATURES = [
    'bm_wind', 'emb_wind', 'solar', 'demand',
    'peak', 'days_ago', 'weekend', 'temp_2m',
    'sin_hour', 'cos_hour',
]

COMMON_MODEL_KWARGS = dict(
    objective='reg:squarederror',
    booster='dart',
    gamma=0.2,
    subsample=1.0,
    n_estimators=200,
    max_depth=10,
    colsample_bytree=1,
)

results_store = {}  # will accumulate all variant results

def run_variant(name, features, train_df, test_df, sample_weights=None):
    """Train XGBoost, evaluate, return per-slot MAE series."""
    X_train = train_df[features].astype(float)
    y_train = train_df['day_ahead'].astype(float)
    X_test = test_df[features].astype(float)
    y_test = test_df['day_ahead'].astype(float)

    if sample_weights is None:
        sw = ((np.log10((y_train - y_train.mean()).abs() + 10) * 5) - 4).round(0)
    else:
        sw = sample_weights

    model = xg.XGBRegressor(**COMMON_MODEL_KWARGS)
    model.fit(X_train, y_train, sample_weight=sw, verbose=False)

    preds = model.predict(X_test)
    errors = np.abs(y_test.values - preds)
    mae_overall = errors.mean()

    result_df = test_df[['time_gb', 'day_ahead']].copy()
    result_df['pred'] = preds
    result_df['error'] = errors
    result_df['bias'] = y_test.values - preds  # positive = underprediction

    mae_evening = result_df[result_df['time_gb'].between(15.5, 22)]['error'].mean()
    mae_overnight = result_df[result_df['time_gb'].between(0, 6)]['error'].mean()
    bias_evening = result_df[result_df['time_gb'].between(15.5, 22)]['bias'].mean()

    print(f'{name:45s}  MAE={mae_overall:.2f}  Evening={mae_evening:.2f} (bias={bias_evening:+.2f})  Overnight={mae_overnight:.2f}')
    results_store[name] = {
        'mae_overall': mae_overall, 'mae_evening': mae_evening,
        'mae_overnight': mae_overnight, 'bias_evening': bias_evening,
        'result_df': result_df, 'model': model,
    }
    return result_df

train_df = df[df['split'] == 'train'].dropna(subset=LEGACY_FEATURES + ['day_ahead'])
test_df = df[df['split'] == 'test'].dropna(subset=LEGACY_FEATURES + ['day_ahead'])
print(f'Train: {len(train_df):,}  Test: {len(test_df):,}')
print()
print(f'{"Variant":<45}  MAE    Evening (bias)  Overnight')
print('-' * 80)
baseline_result = run_variant('baseline (production)', LEGACY_FEATURES, train_df, test_df)

In [ ]:
# MAE by half-hour slot
mae_by_hour = baseline_result.groupby('time_gb')['error'].mean()

fig, ax = plt.subplots(figsize=(16, 5))
colors = ['#d62728' if 15.5 <= t <= 22 else '#1f77b4' for t in mae_by_hour.index]
ax.bar(mae_by_hour.index, mae_by_hour.values, width=0.4, color=colors)
ax.axhline(mae_by_hour.mean(), color='black', linestyle='--', alpha=0.5, label=f'Mean MAE={mae_by_hour.mean():.2f}p')
ax.set_title('Baseline: MAE by hour-of-day (red = evening 15:30-22:00)')
ax.set_xlabel('Hour (GB time)')
ax.set_ylabel('MAE (p/kWh)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Cell 4 — Variant: exponential recency weighting
Replace spike-weighted `sample_weights` with `exp(-days_ago / halflife)`. Try multiple halflives.

In [ ]:
HALFLIVES = [30, 60, 90, 180]

for hl in HALFLIVES:
    sw = np.exp(-train_df['days_ago'].values / hl)
    run_variant(f'recency_decay  halflife={hl}d', LEGACY_FEATURES, train_df, test_df, sw)

# Plot MAE by hour for all recency variants vs baseline
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(mae_by_hour.index, mae_by_hour.values, label='baseline', linewidth=2, color='black')
for hl in HALFLIVES:
    r = results_store[f'recency_decay  halflife={hl}d']['result_df']
    m = r.groupby('time_gb')['error'].mean()
    ax.plot(m.index, m.values, label=f'decay hl={hl}d', alpha=0.8)
ax.axvspan(15.5, 22, alpha=0.08, color='red', label='Evening window')
ax.set_title('MAE by hour: recency decay variants vs baseline')
ax.set_xlabel('Hour (GB time)')
ax.set_ylabel('MAE (p/kWh)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Cell 5 — Variant: drop `days_ago`
`days_ago` is always 0 at inference time — it's a misleading feature. Test without it.

In [ ]:
FEATURES_NO_DAYS_AGO = [f for f in LEGACY_FEATURES if f != 'days_ago']

run_variant('baseline  no_days_ago', FEATURES_NO_DAYS_AGO, train_df, test_df)

# Pick best halflife from cell 4 based on evening MAE
best_hl = min(HALFLIVES, key=lambda hl: results_store[f'recency_decay  halflife={hl}d']['mae_evening'])
print(f'Best halflife from cell 4: {best_hl}d')
sw_best = np.exp(-train_df['days_ago'].values / best_hl)
run_variant(f'recency_decay hl={best_hl}d  no_days_ago', FEATURES_NO_DAYS_AGO, train_df, test_df, sw_best)

## Cell 6 — Variant: recent mean price as regime signal
Add `recent_mean_price` = rolling 7-day mean of actual day-ahead prices.
This tells the model the current market level (captures gas price regime shifts).
**No lookahead**: computed only from actual price history up to each forecast's `created_at`.

In [ ]:
# Build rolling 7-day mean from the prices series (no lookahead into test)
# For each row, recent_mean = mean of day_ahead prices in the 7 days before created_at
prices_series = df.loc[df['split'] == 'train', 'day_ahead'].sort_index()

def compute_recent_mean(row, prices_ts, window_days=7):
    """Mean price in [created_at - window_days, created_at)."""
    end = row['created_at']
    start = end - pd.Timedelta(days=window_days)
    mask = (prices_ts.index >= start) & (prices_ts.index < end)
    vals = prices_ts[mask]
    return vals.mean() if len(vals) > 0 else np.nan

print('Computing recent_mean_price for train set (may take ~30s) ...')
# Use vectorised approach: group by forecast_id, look up window before created_at
prices_ts = df['day_ahead'].sort_index()

fc_means = {}
for fid, grp in df.groupby('forecast_id'):
    ca = grp['created_at'].iloc[0]
    start = ca - pd.Timedelta(days=7)
    mask = (prices_ts.index >= start) & (prices_ts.index < ca)
    fc_means[fid] = prices_ts[mask].mean() if mask.sum() > 0 else np.nan

df['recent_mean_price'] = df['forecast_id'].map(fc_means)

FEATURES_WITH_REGIME = FEATURES_NO_DAYS_AGO + ['recent_mean_price']
train_r = df[df['split'] == 'train'].dropna(subset=FEATURES_WITH_REGIME + ['day_ahead'])
test_r = df[df['split'] == 'test'].dropna(subset=FEATURES_WITH_REGIME + ['day_ahead'])
print(f'With regime feature — train: {len(train_r):,}  test: {len(test_r):,}')

sw_regime = np.exp(-train_r['days_ago'].values / best_hl)
run_variant(f'recency_decay hl={best_hl}d  +recent_mean_price', FEATURES_WITH_REGIME, train_r, test_r, sw_regime)

## Cell 7 — Summary table

In [ ]:
baseline_mae = results_store['baseline (production)']['mae_overall']
baseline_eve = results_store['baseline (production)']['mae_evening']

rows = []
for name, r in results_store.items():
    rows.append({
        'variant': name,
        'MAE overall': round(r['mae_overall'], 3),
        'MAE evening': round(r['mae_evening'], 3),
        'bias evening': round(r['bias_evening'], 3),
        'MAE overnight': round(r['mae_overnight'], 3),
        'Δ overall': round(r['mae_overall'] - baseline_mae, 3),
        'Δ evening': round(r['mae_evening'] - baseline_eve, 3),
    })

summary = pd.DataFrame(rows).sort_values('MAE evening')
summary.to_csv(DATA_DIR / 'results_summary.csv', index=False)

def colour_delta(val):
    if val < -0.1:
        return 'background-color: #c8f7c5'
    elif val > 0.1:
        return 'background-color: #f7c5c5'
    return ''

display(
    summary.style
        .applymap(colour_delta, subset=['Δ overall', 'Δ evening'])
        .format(precision=3)
        .set_caption('Model variants — ranked by evening MAE (green = improvement, red = worse)')
)
print('\nSaved to data/results_summary.csv')